In [1]:

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np

In [3]:
ratings = pd.read_csv(
    "/content/drive/MyDrive/HYBRID RECOMMENDER SYSTEM/ratings.dat",
    sep="::",
    engine="python",
    names=["userId", "movieId", "rating", "timestamp"]
)

movies_ml = pd.read_csv(
    "/content/drive/MyDrive/HYBRID RECOMMENDER SYSTEM/movies.dat",
    sep="::",
    engine="python",
    names=["movieId", "title", "genres"],
    encoding="latin-1"
)

In [4]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [5]:
movies_ml.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3883 entries, 0 to 3882
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  3883 non-null   int64 
 1   title    3883 non-null   object
 2   genres   3883 non-null   object
dtypes: int64(1), object(2)
memory usage: 91.1+ KB


In [6]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 4 columns):
 #   Column     Non-Null Count    Dtype
---  ------     --------------    -----
 0   userId     1000209 non-null  int64
 1   movieId    1000209 non-null  int64
 2   rating     1000209 non-null  int64
 3   timestamp  1000209 non-null  int64
dtypes: int64(4)
memory usage: 30.5 MB


In [7]:
movies_meta = pd.read_csv(
    "/content/drive/MyDrive/HYBRID RECOMMENDER SYSTEM/movies_metadata.xls",
    low_memory=False
)
# Keep only valid TMDB ids
movies_meta = movies_meta[movies_meta["id"].str.isnumeric()]
movies_meta["id"] = movies_meta["id"].astype(int)


In [8]:
movies_meta.info()

<class 'pandas.core.frame.DataFrame'>
Index: 45463 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45463 non-null  object 
 1   belongs_to_collection  4491 non-null   object 
 2   budget                 45463 non-null  object 
 3   genres                 45463 non-null  object 
 4   homepage               7779 non-null   object 
 5   id                     45463 non-null  int64  
 6   imdb_id                45446 non-null  object 
 7   original_language      45452 non-null  object 
 8   original_title         45463 non-null  object 
 9   overview               44509 non-null  object 
 10  popularity             45460 non-null  object 
 11  poster_path            45077 non-null  object 
 12  production_companies   45460 non-null  object 
 13  production_countries   45460 non-null  object 
 14  release_date           45376 non-null  object 
 15  revenue

In [9]:
movies_meta.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [10]:
credits = pd.read_csv('/content/drive/MyDrive/HYBRID RECOMMENDER SYSTEM/credits.xls')
keywords = pd.read_csv('/content/drive/MyDrive/HYBRID RECOMMENDER SYSTEM/keywords.xls')
tmdb = movies_meta.merge(credits, on="id") \
                   .merge(keywords, on="id")


In [11]:
credits.head()

,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...",11862


In [12]:
keywords.head()

,id,keywords
0,862,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,8844,"[{'id': 10090, 'name': 'board game'}, {'id': 1..."
2,15602,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392..."
3,31357,"[{'id': 818, 'name': 'based on novel'}, {'id':..."
4,11862,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n..."


In [13]:

tmdb.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count', 'cast', 'crew', 'keywords'],
      dtype='object')

In [14]:
tmdb["homepage"]

,homepage
0,http://toystory.disney.com/toy-story
1,NaN
2,NaN
3,NaN
4,NaN
...,...
46623,http://www.imdb.com/title/tt6209470/
46624,NaN
46625,NaN
46626,NaN


In [15]:
tmdb["poster_path"]

,poster_path
0,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg
1,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg
2,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg
3,/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg
4,/e64sOI48hQXyru7naBFyssKFxVd.jpg
...,...
46623,/jldsYflnId4tTWPx8es3uzsB1I8.jpg
46624,/xZkmxsNmYXJbKVsTRLLx3pqGHx7.jpg
46625,/d5bX92nDsISNhu3ZT69uHwmfCGw.jpg
46626,/aorBPO7ak8e8iJKT5OcqYxU3jlK.jpg


In [16]:
links = pd.read_csv(
    "/content/drive/MyDrive/HYBRID RECOMMENDER SYSTEM/links.xls"
)
# Clean ids
links = links.dropna(subset=["tmdbId", "movieId"])
links["tmdbId"] = links["tmdbId"].astype(int)


In [17]:
links.head()

,movieId,imdbId,tmdbId
0,1,114709,862
1,2,113497,8844
2,3,113228,15602
3,4,114885,31357
4,5,113041,11862


In [18]:
links.info()

<class 'pandas.core.frame.DataFrame'>
Index: 45624 entries, 0 to 45842
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  45624 non-null  int64
 1   imdbId   45624 non-null  int64
 2   tmdbId   45624 non-null  int64
dtypes: int64(3)
memory usage: 1.4 MB


In [19]:
movies_common = movies_ml.merge(
    links[["movieId", "tmdbId"]],
    on="movieId",
    how="inner"
).merge(
    tmdb,
    left_on="tmdbId",
    right_on="id",
    how="inner"
)


In [20]:
print("MovieLens movies:", movies_ml.movieId.nunique())
print("Common movies:", movies_common.movieId.nunique())


MovieLens movies: 3883
Common movies: 3828


In [21]:
ratings.shape

(1000209, 4)

In [22]:
common_movie_ids = set(movies_common.movieId)

ratings_clean = ratings[
    ratings.movieId.isin(common_movie_ids)
]


In [23]:
ratings.shape

(1000209, 4)

In [24]:
ratings_clean.shape


(997407, 4)

In [25]:
ratings_clean

,userId,movieId,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,5,956704887
1000206,6040,562,5,956704746
1000207,6040,1096,4,956715648


In [26]:
movies_common.isnull().sum()

,0
movieId,0
title_x,0
genres_x,0
tmdbId,0
adult,0
belongs_to_collection,3194
budget,0
genres_y,0
homepage,3623
id,0


In [83]:
content_df = movies_common[[
    "movieId",
    "title_x",
    "overview",
    "genres_x",
    "keywords",
    "cast",
    "crew"
]].copy()


In [84]:
content_df.isnull().sum()

,0
movieId,0
title_x,0
overview,18
genres_x,0
keywords,0
cast,0
crew,0


In [85]:
content_df.rename(columns={"title_x":"title","genres_x":"genres"},inplace=True)

In [86]:
content_df.dropna(inplace=True)

In [87]:
content_df

,movieId,title,overview,genres,keywords,cast,crew
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation|Children's|Comedy,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...","[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de..."
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure|Children's|Fantasy,"[{'id': 10090, 'name': 'board game'}, {'id': 1...","[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de..."
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy|Romance,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392...","[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de..."
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy|Drama,"[{'id': 818, 'name': 'based on novel'}, {'id':...","[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de..."
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n...","[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de..."
...,...,...,...,...,...,...,...
3858,3948,Meet the Parents (2000),"Greg Focker is ready to marry his girlfriend, ...",Comedy,"[{'id': 591, 'name': 'cia'}, {'id': 822, 'name...","[{'cast_id': 1, 'character': 'Gaylord ""Greg"" F...","[{'credit_id': '52fe4303c3a36847f8033ca9', 'de..."
3859,3949,Requiem for a Dream (2000),The hopes and dreams of four ambitious people ...,Drama,"[{'id': 1803, 'name': 'drug addiction'}, {'id'...","[{'cast_id': 29, 'character': 'Sara Goldfarb',...","[{'credit_id': '52fe4263c3a36847f801a72b', 'de..."
3860,3950,Tigerland (2000),A group of recruits go through Advanced Infant...,Drama,"[{'id': 10183, 'name': 'independent film'}, {'...","[{'cast_id': 1, 'character': 'Pvt. Roland Bozz...","[{'credit_id': '52fe43a39251416c75018317', 'de..."
3861,3951,Two Family House (2000),Buddy Visalo (Michael Rispoli) is a factory wo...,Drama,[],"[{'cast_id': 1, 'character': 'Buddy Visalo', '...","[{'credit_id': '52fe46c2c3a368484e0a2471', 'de..."


In [88]:
content_df["genres"]=content_df["genres"].str.replace("|"," ")

In [89]:
content_df

,movieId,title,overview,genres,keywords,cast,crew
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Children's Comedy,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...","[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de..."
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Children's Fantasy,"[{'id': 10090, 'name': 'board game'}, {'id': 1...","[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de..."
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy Romance,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392...","[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de..."
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama,"[{'id': 818, 'name': 'based on novel'}, {'id':...","[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de..."
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n...","[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de..."
...,...,...,...,...,...,...,...
3858,3948,Meet the Parents (2000),"Greg Focker is ready to marry his girlfriend, ...",Comedy,"[{'id': 591, 'name': 'cia'}, {'id': 822, 'name...","[{'cast_id': 1, 'character': 'Gaylord ""Greg"" F...","[{'credit_id': '52fe4303c3a36847f8033ca9', 'de..."
3859,3949,Requiem for a Dream (2000),The hopes and dreams of four ambitious people ...,Drama,"[{'id': 1803, 'name': 'drug addiction'}, {'id'...","[{'cast_id': 29, 'character': 'Sara Goldfarb',...","[{'credit_id': '52fe4263c3a36847f801a72b', 'de..."
3860,3950,Tigerland (2000),A group of recruits go through Advanced Infant...,Drama,"[{'id': 10183, 'name': 'independent film'}, {'...","[{'cast_id': 1, 'character': 'Pvt. Roland Bozz...","[{'credit_id': '52fe43a39251416c75018317', 'de..."
3861,3951,Two Family House (2000),Buddy Visalo (Michael Rispoli) is a factory wo...,Drama,[],"[{'cast_id': 1, 'character': 'Buddy Visalo', '...","[{'credit_id': '52fe46c2c3a368484e0a2471', 'de..."


In [90]:
import ast


In [91]:
def extract_names(text):
    data=ast.literal_eval(text)
    return " ".join([d["name"] for d in data])


In [92]:
content_df["keywords"][0]

"[{'id': 931, 'name': 'jealousy'}, {'id': 4290, 'name': 'toy'}, {'id': 5202, 'name': 'boy'}, {'id': 6054, 'name': 'friendship'}, {'id': 9713, 'name': 'friends'}, {'id': 9823, 'name': 'rivalry'}, {'id': 165503, 'name': 'boy next door'}, {'id': 170722, 'name': 'new toy'}, {'id': 187065, 'name': 'toy comes to life'}]"

In [93]:
extract_names(content_df["keywords"][0])

'jealousy toy boy friendship friends rivalry boy next door new toy toy comes to life'

In [94]:
content_df["keywords"] = content_df["keywords"].apply(extract_names)



In [95]:
content_df

,movieId,title,overview,genres,keywords,cast,crew
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Children's Comedy,jealousy toy boy friendship friends rivalry bo...,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de..."
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Children's Fantasy,board game disappearance based on children's b...,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de..."
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy Romance,fishing best friend duringcreditsstinger old men,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de..."
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama,based on novel interracial relationship single...,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de..."
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,baby midlife crisis confidence aging daughter ...,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de..."
...,...,...,...,...,...,...,...
3858,3948,Meet the Parents (2000),"Greg Focker is ready to marry his girlfriend, ...",Comedy,cia airport cat jew orderly airplane father-in...,"[{'cast_id': 1, 'character': 'Gaylord ""Greg"" F...","[{'credit_id': '52fe4303c3a36847f8033ca9', 'de..."
3859,3949,Requiem for a Dream (2000),The hopes and dreams of four ambitious people ...,Drama,drug addiction junkie heroin speed diet unsoci...,"[{'cast_id': 29, 'character': 'Sara Goldfarb',...","[{'credit_id': '52fe4263c3a36847f801a72b', 'de..."
3860,3950,Tigerland (2000),A group of recruits go through Advanced Infant...,Drama,independent film awol fort polk louisiana targ...,"[{'cast_id': 1, 'character': 'Pvt. Roland Bozz...","[{'credit_id': '52fe43a39251416c75018317', 'de..."
3861,3951,Two Family House (2000),Buddy Visalo (Michael Rispoli) is a factory wo...,Drama,,"[{'cast_id': 1, 'character': 'Buddy Visalo', '...","[{'credit_id': '52fe46c2c3a368484e0a2471', 'de..."


In [96]:
content_df["cast"][0]

"[{'cast_id': 14, 'character': 'Woody (voice)', 'credit_id': '52fe4284c3a36847f8024f95', 'gender': 2, 'id': 31, 'name': 'Tom Hanks', 'order': 0, 'profile_path': '/pQFoyx7rp09CJTAb932F2g8Nlho.jpg'}, {'cast_id': 15, 'character': 'Buzz Lightyear (voice)', 'credit_id': '52fe4284c3a36847f8024f99', 'gender': 2, 'id': 12898, 'name': 'Tim Allen', 'order': 1, 'profile_path': '/uX2xVf6pMmPepxnvFWyBtjexzgY.jpg'}, {'cast_id': 16, 'character': 'Mr. Potato Head (voice)', 'credit_id': '52fe4284c3a36847f8024f9d', 'gender': 2, 'id': 7167, 'name': 'Don Rickles', 'order': 2, 'profile_path': '/h5BcaDMPRVLHLDzbQavec4xfSdt.jpg'}, {'cast_id': 17, 'character': 'Slinky Dog (voice)', 'credit_id': '52fe4284c3a36847f8024fa1', 'gender': 2, 'id': 12899, 'name': 'Jim Varney', 'order': 3, 'profile_path': '/eIo2jVVXYgjDtaHoF19Ll9vtW7h.jpg'}, {'cast_id': 18, 'character': 'Rex (voice)', 'credit_id': '52fe4284c3a36847f8024fa5', 'gender': 2, 'id': 12900, 'name': 'Wallace Shawn', 'order': 4, 'profile_path': '/oGE6JqPP2xH4t

In [97]:
def convert_cast(text):
    return " ".join([d["name"] for d in ast.literal_eval(text)][:3])

In [98]:
content_df["cast"] = content_df["cast"].apply(convert_cast)

In [99]:
content_df.head()

,movieId,title,overview,genres,keywords,cast,crew
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Children's Comedy,jealousy toy boy friendship friends rivalry bo...,Tom Hanks Tim Allen Don Rickles,"[{'credit_id': '52fe4284c3a36847f8024f49', 'de..."
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Children's Fantasy,board game disappearance based on children's b...,Robin Williams Jonathan Hyde Kirsten Dunst,"[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de..."
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy Romance,fishing best friend duringcreditsstinger old men,Walter Matthau Jack Lemmon Ann-Margret,"[{'credit_id': '52fe466a9251416c75077a89', 'de..."
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama,based on novel interracial relationship single...,Whitney Houston Angela Bassett Loretta Devine,"[{'credit_id': '52fe44779251416c91011acb', 'de..."
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,baby midlife crisis confidence aging daughter ...,Steve Martin Diane Keaton Martin Short,"[{'credit_id': '52fe44959251416c75039ed7', 'de..."


In [100]:
content_df["crew"][0]

'[{\'credit_id\': \'52fe4284c3a36847f8024f49\', \'department\': \'Directing\', \'gender\': 2, \'id\': 7879, \'job\': \'Director\', \'name\': \'John Lasseter\', \'profile_path\': \'/7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg\'}, {\'credit_id\': \'52fe4284c3a36847f8024f4f\', \'department\': \'Writing\', \'gender\': 2, \'id\': 12891, \'job\': \'Screenplay\', \'name\': \'Joss Whedon\', \'profile_path\': \'/dTiVsuaTVTeGmvkhcyJvKp2A5kr.jpg\'}, {\'credit_id\': \'52fe4284c3a36847f8024f55\', \'department\': \'Writing\', \'gender\': 2, \'id\': 7, \'job\': \'Screenplay\', \'name\': \'Andrew Stanton\', \'profile_path\': \'/pvQWsu0qc8JFQhMVJkTHuexUAa1.jpg\'}, {\'credit_id\': \'52fe4284c3a36847f8024f5b\', \'department\': \'Writing\', \'gender\': 2, \'id\': 12892, \'job\': \'Screenplay\', \'name\': \'Joel Cohen\', \'profile_path\': \'/dAubAiZcvKFbboWlj7oXOkZnTSu.jpg\'}, {\'credit_id\': \'52fe4284c3a36847f8024f61\', \'department\': \'Writing\', \'gender\': 0, \'id\': 12893, \'job\': \'Screenplay\', \'name\': \'A

In [101]:
def convert_crew(text):
    return " ".join([d["name"] for d in ast.literal_eval(text) if d['job']=="Director"])

In [102]:
content_df["crew"] = content_df["crew"].apply(convert_crew)

In [103]:
content_df.head()

,movieId,title,overview,genres,keywords,cast,crew
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Children's Comedy,jealousy toy boy friendship friends rivalry bo...,Tom Hanks Tim Allen Don Rickles,John Lasseter
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Children's Fantasy,board game disappearance based on children's b...,Robin Williams Jonathan Hyde Kirsten Dunst,Joe Johnston
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy Romance,fishing best friend duringcreditsstinger old men,Walter Matthau Jack Lemmon Ann-Margret,Howard Deutch
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama,based on novel interracial relationship single...,Whitney Houston Angela Bassett Loretta Devine,Forest Whitaker
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,baby midlife crisis confidence aging daughter ...,Steve Martin Diane Keaton Martin Short,Charles Shyer


In [104]:
# Clean genres
content_df["genres_clean"] = (
    content_df["genres"]
    .str.replace("|", " ", regex=False)
    .str.lower()
)

# Clean keywords (already space separated usually)
content_df["keywords_clean"] = (
    content_df["keywords"]
    .str.lower()
)

# Final explainable text
# content_df["explainable_text"] = (
#     content_df["genres_clean"] + " " +
#     content_df["keywords_clean"]
# )
# content_df["combined_features"] = (
#     (content_df["genres_clean"] + " ") * 3 +   # weight genres
#     content_df["keywords_clean"] + " " +
#     content_df["cast"] + " " +
#     content_df["crew"]
# )




In [105]:
import re
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [106]:
def clean_text(text):
    if pd.isna(text):
        return ""

    # lowercase
    text = text.lower()

    # remove special characters & numbers
    text = re.sub(r"[^a-z\s]", " ", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # remove stopwords
    words = text.split()
    words = [w for w in words if w not in ENGLISH_STOP_WORDS]

    return " ".join(words)

In [107]:
content_df["overview_clean"] = content_df["overview"].apply(clean_text)

In [108]:
content_df["overview_clean"] = content_df["overview"].apply(clean_text)

In [109]:
content_df["combined_features"] = (
    (content_df["genres_clean"] + " ") * 3 +   # weight genres
    content_df["keywords_clean"] + " " +
    content_df["cast"] + " " +
    content_df["crew"]+" "+content_df["overview_clean"]
)

In [110]:
content_df.head()

,movieId,title,overview,genres,keywords,cast,crew,genres_clean,keywords_clean,overview_clean,combined_features
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Children's Comedy,jealousy toy boy friendship friends rivalry bo...,Tom Hanks Tim Allen Don Rickles,John Lasseter,animation children's comedy,jealousy toy boy friendship friends rivalry bo...,led woody andy s toys live happily room andy s...,animation children's comedy animation children...
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Children's Fantasy,board game disappearance based on children's b...,Robin Williams Jonathan Hyde Kirsten Dunst,Joe Johnston,adventure children's fantasy,board game disappearance based on children's b...,siblings judy peter discover enchanted board g...,adventure children's fantasy adventure childre...
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy Romance,fishing best friend duringcreditsstinger old men,Walter Matthau Jack Lemmon Ann-Margret,Howard Deutch,comedy romance,fishing best friend duringcreditsstinger old men,family wedding reignites ancient feud door nei...,comedy romance comedy romance comedy romance f...
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama,based on novel interracial relationship single...,Whitney Houston Angela Bassett Loretta Devine,Forest Whitaker,comedy drama,based on novel interracial relationship single...,cheated mistreated stepped women holding breat...,comedy drama comedy drama comedy drama based o...
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,baby midlife crisis confidence aging daughter ...,Steve Martin Diane Keaton Martin Short,Charles Shyer,comedy,baby midlife crisis confidence aging daughter ...,just george banks recovered daughter s wedding...,comedy comedy comedy baby midlife crisis confi...


In [111]:
content_df.head()

,movieId,title,overview,genres,keywords,cast,crew,genres_clean,keywords_clean,overview_clean,combined_features
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Children's Comedy,jealousy toy boy friendship friends rivalry bo...,Tom Hanks Tim Allen Don Rickles,John Lasseter,animation children's comedy,jealousy toy boy friendship friends rivalry bo...,led woody andy s toys live happily room andy s...,animation children's comedy animation children...
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Children's Fantasy,board game disappearance based on children's b...,Robin Williams Jonathan Hyde Kirsten Dunst,Joe Johnston,adventure children's fantasy,board game disappearance based on children's b...,siblings judy peter discover enchanted board g...,adventure children's fantasy adventure childre...
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy Romance,fishing best friend duringcreditsstinger old men,Walter Matthau Jack Lemmon Ann-Margret,Howard Deutch,comedy romance,fishing best friend duringcreditsstinger old men,family wedding reignites ancient feud door nei...,comedy romance comedy romance comedy romance f...
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama,based on novel interracial relationship single...,Whitney Houston Angela Bassett Loretta Devine,Forest Whitaker,comedy drama,based on novel interracial relationship single...,cheated mistreated stepped women holding breat...,comedy drama comedy drama comedy drama based o...
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,baby midlife crisis confidence aging daughter ...,Steve Martin Diane Keaton Martin Short,Charles Shyer,comedy,baby midlife crisis confidence aging daughter ...,just george banks recovered daughter s wedding...,comedy comedy comedy baby midlife crisis confi...


In [57]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf_sim = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)


tfidf_matrix_sim = tfidf_sim.fit_transform(
    content_df["combined_features"]
)


In [58]:
cosine_sim = cosine_similarity(
    tfidf_matrix_sim,
    tfidf_matrix_sim
)


In [59]:
movie_index = pd.Series(
    content_df.index,
    index=content_df["movieId"]
)


In [60]:
def recommend_movies_content(movie_id, top_n=10):
    if movie_id not in movie_index:
        return "Movie not found"

    idx = movie_index[movie_id]

    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n + 1]

    movie_indices = [i[0] for i in similarity_scores]

    return content_df.iloc[movie_indices][["movieId", "title"]]


In [61]:
content_df[content_df["title"] == "Toy Story (1995)"][["movieId", "title"]]


,movieId,title
0,1,Toy Story (1995)


In [62]:
recommend_movies_content(movie_id=1	)


,movieId,title
3032,3114,Toy Story 2 (1999)
1839,1920,Small Soldiers (1998)
1055,1064,Aladdin and the King of Thieves (1996)
2060,2141,"American Tail, An (1986)"
2061,2142,"American Tail: Fievel Goes West, An (1991)"
2270,2355,"Bug's Life, A (1998)"
3667,3754,"Adventures of Rocky and Bullwinkle, The (2000)"
3524,3611,Saludos Amigos (1943)
2269,2354,"Rugrats Movie, The (1998)"
581,588,Aladdin (1992)


In [63]:
content_df.sample(10)

,movieId,title,combined_features,genres_clean,keywords_clean
1241,1265,Groundhog Day (1993),comedy romance comedy romance comedy romance d...,comedy romance,deja vu groundhog weather forecast telecaster ...
1111,1125,"Return of the Pink Panther, The (1974)",comedy comedy comedy robbery diamant côte d'az...,comedy,robbery diamant côte d'azur inspector panther
893,906,Gaslight (1944),mystery thriller mystery thriller mystery thri...,mystery thriller,italy scotland yard letter aunt victorian engl...
849,864,"Wife, The (1995)",comedy drama comedy drama comedy drama based o...,comedy drama,based on novel wife marriage unhappiness
864,876,Police Story 4: Project S (Chao ji ji hua) (1993),action action action martial arts Michelle Yeo...,action,martial arts
2971,3053,"Messenger: The Story of Joan of Arc, The (1999)",drama war drama war drama war schizophrenia fr...,drama war,schizophrenia france rape siege biography orlé...
236,240,Hideaway (1995),thriller thriller thriller suspense Jeff Goldb...,thriller,suspense
1443,1482,"Van, The (1996)",comedy drama comedy drama comedy drama male fr...,comedy drama,male friendship van snack bar soccer unemployment
2471,2556,Telling You (1998),comedy drama romance comedy drama romance come...,comedy drama romance,
109,111,Taxi Driver (1976),drama thriller drama thriller drama thriller v...,drama thriller,vietnam veteran taxi obsession drug dealer nig...


In [64]:
ratings_clean


,userId,movieId,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,5,956704887
1000206,6040,562,5,956704746
1000207,6040,1096,4,956715648


In [68]:
!pip install numpy==1.26.4

In [66]:
!pip install scikit-surprise

In [67]:
import surprise
print(surprise.__version__)


1.1.4


In [69]:
!pip install scikit-surprise


In [70]:
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split
import math


In [71]:
reader = Reader(rating_scale=(0.5, 5.0))

data = Dataset.load_from_df(
    ratings_clean[["userId", "movieId", "rating"]],
    reader
)


In [122]:
ratings_clean.head()

,userId,movieId,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [72]:
svd = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)


In [73]:
trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

svd.fit(trainset)


In [74]:
def cf_score(user_id, movie_id):
    return svd.predict(user_id, movie_id).est


In [75]:
def rank_movies_cf(user_id, candidate_movie_ids, top_n=10):
    scores = []

    for movie_id in candidate_movie_ids:
        score = cf_score(user_id, movie_id)
        scores.append((movie_id, score))

    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_n]


In [76]:
predictions = svd.test(testset)




In [77]:
# Step 1: get candidates from content-based model
content_candidates = recommend_movies_content(movie_id=1, top_n=10)["movieId"]

# Step 2: rank them using CF
ranked = rank_movies_cf(user_id=10, candidate_movie_ids=content_candidates)

# Step 3: show ranked movies
ranked_movies = pd.DataFrame(ranked, columns=["movieId", "cf_score"])
ranked_movies.merge(
    content_df[["movieId", "title"]],
    on="movieId"
)


,movieId,cf_score,title
0,588,4.857705,Aladdin (1992)
1,3114,4.349145,Toy Story 2 (1999)
2,2141,4.223761,"American Tail, An (1986)"
3,2355,4.212605,"Bug's Life, A (1998)"
4,2142,3.892748,"American Tail: Fievel Goes West, An (1991)"
5,1920,3.819758,Small Soldiers (1998)
6,3611,3.662182,Saludos Amigos (1943)
7,1064,3.613853,Aladdin and the King of Thieves (1996)
8,3754,3.491958,"Adventures of Rocky and Bullwinkle, The (2000)"
9,2354,3.335601,"Rugrats Movie, The (1998)"


In [127]:
def get_set(series_value):
    if isinstance(series_value, str):
        return set(series_value.lower().split())
    return set()


In [128]:
def generate_rich_explanation(seed_movie_id, candidate_movie_id):
    # Seed movie data
    seed_row = content_df[content_df["movieId"] == seed_movie_id].iloc[0]
    cand_row = content_df[content_df["movieId"] == candidate_movie_id].iloc[0]

    seed_title = seed_row["title"]

    # Genres
    seed_genres = get_set(seed_row["genres_clean"])
    cand_genres = get_set(cand_row["genres_clean"])
    shared_genres = seed_genres.intersection(cand_genres)

    # Cast
    seed_cast = get_set(seed_row["cast"])
    cand_cast = get_set(cand_row["cast"])
    shared_cast = seed_cast.intersection(cand_cast)

    # Director
    seed_director = seed_row.get("crew", "")
    cand_director = cand_row.get("crew", "")
    same_director = (
        isinstance(seed_director, str)
        and isinstance(cand_director, str)
        and seed_director.lower() == cand_director.lower()
        and seed_director != ""
    )

    reasons = []

    if shared_genres:
        reasons.append(
            "shared genres like " +
            ", ".join([g.title() for g in list(shared_genres)[:2]])
        )

    if shared_cast:
        reasons.append(
            "common cast members such as " +
            ", ".join([c.title() for c in list(shared_cast)[:2]])
        )

    if same_director:
        reasons.append(
            f"the same director ({seed_director})"
        )

    if not reasons:
        return (
            f"Recommended because it is similar in overall theme to "
            f"'{seed_title}'."
        )

    return (
        f"Recommended because it shares "
        + ", and ".join(reasons)
        + f" with '{seed_title}'."
    )


In [129]:
seed_movie_id = 1

content_candidates = recommend_movies_content(
    movie_id=seed_movie_id,
    top_n=10
)

explanations = {}

for movie_id in content_candidates["movieId"]:
    explanations[movie_id] = generate_rich_explanation(
        seed_movie_id,
        movie_id
    )


In [130]:
ranked = rank_movies_cf(
    user_id=10,
    candidate_movie_ids=content_candidates["movieId"].tolist()
)

ranked_df = (
    pd.DataFrame(ranked, columns=["movieId", "cf_score"])
    .merge(content_df[["movieId", "title"]], on="movieId")
)


In [131]:
ranked_df["explanation"] = ranked_df["movieId"].apply(
    lambda x: explanations[x]
)

ranked_df


,movieId,cf_score,title,explanation
0,588,4.857705,Aladdin (1992),Recommended because it shares shared genres li...
1,3114,4.349145,Toy Story 2 (1999),Recommended because it shares shared genres li...
2,2141,4.223761,"American Tail, An (1986)",Recommended because it shares shared genres li...
3,2355,4.212605,"Bug's Life, A (1998)",Recommended because it shares shared genres li...
4,2142,3.892748,"American Tail: Fievel Goes West, An (1991)",Recommended because it shares shared genres li...
5,1920,3.819758,Small Soldiers (1998),Recommended because it shares shared genres li...
6,3611,3.662182,Saludos Amigos (1943),Recommended because it shares shared genres li...
7,1064,3.613853,Aladdin and the King of Thieves (1996),Recommended because it shares shared genres li...
8,3754,3.491958,"Adventures of Rocky and Bullwinkle, The (2000)",Recommended because it shares shared genres li...
9,2354,3.335601,"Rugrats Movie, The (1998)",Recommended because it shares shared genres li...


In [132]:
ranked_df["explanation"][8]

"Recommended because it shares shared genres like Comedy, Children'S with 'Toy Story (1995)'."

In [133]:
content_df

,movieId,title,overview,genres,keywords,cast,crew,genres_clean,keywords_clean,overview_clean,combined_features
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Children's Comedy,jealousy toy boy friendship friends rivalry bo...,Tom Hanks Tim Allen Don Rickles,John Lasseter,animation children's comedy,jealousy toy boy friendship friends rivalry bo...,led woody andy s toys live happily room andy s...,animation children's comedy animation children...
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Children's Fantasy,board game disappearance based on children's b...,Robin Williams Jonathan Hyde Kirsten Dunst,Joe Johnston,adventure children's fantasy,board game disappearance based on children's b...,siblings judy peter discover enchanted board g...,adventure children's fantasy adventure childre...
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy Romance,fishing best friend duringcreditsstinger old men,Walter Matthau Jack Lemmon Ann-Margret,Howard Deutch,comedy romance,fishing best friend duringcreditsstinger old men,family wedding reignites ancient feud door nei...,comedy romance comedy romance comedy romance f...
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama,based on novel interracial relationship single...,Whitney Houston Angela Bassett Loretta Devine,Forest Whitaker,comedy drama,based on novel interracial relationship single...,cheated mistreated stepped women holding breat...,comedy drama comedy drama comedy drama based o...
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,baby midlife crisis confidence aging daughter ...,Steve Martin Diane Keaton Martin Short,Charles Shyer,comedy,baby midlife crisis confidence aging daughter ...,just george banks recovered daughter s wedding...,comedy comedy comedy baby midlife crisis confi...
...,...,...,...,...,...,...,...,...,...,...,...
3858,3948,Meet the Parents (2000),"Greg Focker is ready to marry his girlfriend, ...",Comedy,cia airport cat jew orderly airplane father-in...,Ben Stiller Robert De Niro Teri Polo,Jay Roach,comedy,cia airport cat jew orderly airplane father-in...,greg focker ready marry girlfriend pam pops qu...,comedy comedy comedy cia airport cat jew order...
3859,3949,Requiem for a Dream (2000),The hopes and dreams of four ambitious people ...,Drama,drug addiction junkie heroin speed diet unsoci...,Ellen Burstyn Jared Leto Jennifer Connelly,Darren Aronofsky,drama,drug addiction junkie heroin speed diet unsoci...,hopes dreams ambitious people shattered drug a...,drama drama drama drug addiction junkie heroin...
3860,3950,Tigerland (2000),A group of recruits go through Advanced Infant...,Drama,independent film awol fort polk louisiana targ...,Colin Farrell Matthew Davis Clifton Collins Jr,Joel Schumacher,drama,independent film awol fort polk louisiana targ...,group recruits advanced infantry training fort...,drama drama drama independent film awol fort p...
3861,3951,Two Family House (2000),Buddy Visalo (Michael Rispoli) is a factory wo...,Drama,,Michael Rispoli Kelly Macdonald Kathrine Narducci,Raymond De Felitta,drama,,buddy visalo michael rispoli factory worker fr...,drama drama drama Michael Rispoli Kelly Macdo...


In [134]:
rmse = accuracy.rmse(predictions)

RMSE: 0.8737


In [135]:
def get_relevant_items(testset, threshold=4.0):
    relevant = {}

    for uid, iid, true_r in testset:
        if true_r >= threshold:
            relevant.setdefault(uid, set()).add(iid)

    return relevant

relevant_items = get_relevant_items(testset)




In [136]:
def get_top_k_predictions(model, testset, k=10):
    user_predictions = {}

    for uid, iid, true_r in testset:
        est = model.predict(uid, iid).est
        user_predictions.setdefault(uid, []).append((iid, est))

    # Sort predictions for each user
    for uid in user_predictions:
        user_predictions[uid].sort(key=lambda x: x[1], reverse=True)
        user_predictions[uid] = user_predictions[uid][:k]

    return user_predictions

top_k_preds = get_top_k_predictions(svd, testset, k=10)


In [137]:
def precision_recall_at_k(relevant, predicted, k=10):
    precisions = []
    recalls = []

    for uid in relevant:
        if uid not in predicted:
            continue

        pred_items = {iid for iid, _ in predicted[uid]}
        rel_items = relevant[uid]

        tp = len(pred_items & rel_items)

        precisions.append(tp / k)
        recalls.append(tp / len(rel_items))

    return (
        sum(precisions) / len(precisions),
        sum(recalls) / len(recalls)
    )

precision, recall = precision_recall_at_k(
    relevant_items,
    top_k_preds,
    k=10
)

print("Precision@10:", precision)
print("Recall@10:", recall)


Precision@10: 0.6850425212606303
Recall@10: 0.641410068125028


In [138]:
def ndcg_at_k(relevant, predicted, k=10):
    ndcgs = []

    for uid in relevant:
        if uid not in predicted:
            continue

        dcg = 0.0
        for rank, (iid, _) in enumerate(predicted[uid], start=1):
            if iid in relevant[uid]:
                dcg += 1 / math.log2(rank + 1)

        ideal_dcg = sum(
            1 / math.log2(rank + 1)
            for rank in range(1, min(len(relevant[uid]), k) + 1)
        )

        if ideal_dcg > 0:
            ndcgs.append(dcg / ideal_dcg)

    return sum(ndcgs) / len(ndcgs)

ndcg = ndcg_at_k(
    relevant_items,
    top_k_preds,
    k=10
)

print("NDCG@10:", ndcg)


NDCG@10: 0.873798409954208
